<a href="https://colab.research.google.com/github/Kunalzzxx/Data-Reconciliation-Modelling-Results-vs-Actual-Outcomes-/blob/main/Data_Reconciliation_Modelling_Results_vs_Actual_Outcomes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""
COVID-19 Forecast Reconciliation Analysis
Project: Recalculated Modelling Results vs Actual Outcomes
Skills: Pandas, NumPy, Matplotlib, Seaborn

HOW TO RUN:
  1. Run generate_data.py FIRST — it creates covid_reconciliation_data/covid_forecast_actuals.csv
  2. Then run this script

  Works in Google Colab, Jupyter Notebook, and local Python.
  All output charts saved to covid_reconciliation_data/outputs/
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")   # needed for Colab / headless environments
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# PATHS — portable, works anywhere
# ─────────────────────────────────────────────
BASE_DIR   = "covid_reconciliation_data"
DATA_PATH  = os.path.join(BASE_DIR, "covid_forecast_actuals.csv")
OUT_DIR    = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# CHART STYLE
# ─────────────────────────────────────────────
PALETTE  = ["#00C2CB", "#FF6B6B", "#FFD166", "#06D6A0", "#A78BFA"]
BG_COLOR = "#0D1117"
TEXT_COL = "#E6EDF3"
GRID_COL = "#21262D"

plt.rcParams.update({
    "figure.facecolor": BG_COLOR, "axes.facecolor":  BG_COLOR,
    "axes.edgecolor":   GRID_COL, "axes.labelcolor": TEXT_COL,
    "xtick.color":      TEXT_COL, "ytick.color":     TEXT_COL,
    "text.color":       TEXT_COL, "grid.color":      GRID_COL,
    "grid.linewidth":   0.5,      "font.family":     "monospace",
    "axes.titlesize":   11,       "axes.labelsize":  9,
})

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv("covid_forecast_actuals.csv", parse_dates=["date"])

# ─────────────────────────────────────────────
# RECONCILIATION METRICS
# ─────────────────────────────────────────────
def reconciliation_metrics(df):
    grp = df.groupby("country").agg(
        total_actual      = ("actual_cases",   "sum"),
        total_forecast    = ("forecast_cases", "sum"),
        mean_abs_error    = ("abs_error",      "mean"),
        mean_variance_pct = ("variance_pct",   "mean"),
        weeks             = ("date",           "count"),
    ).reset_index()
    grp["total_variance"]     = grp["total_forecast"] - grp["total_actual"]
    grp["total_variance_pct"] = ((grp["total_variance"] / grp["total_actual"]) * 100).round(2)
    grp["MAE"]                = grp["mean_abs_error"].round(0).astype(int)
    mape_series = (
        df.groupby("country")["abs_error"].mean() /
        df.groupby("country")["actual_cases"].mean() * 100
    ).round(2)
    grp["MAPE"] = grp["country"].map(mape_series)
    return grp

metrics = reconciliation_metrics(df)
print("\n" + "=" * 65)
print("  RECONCILIATION SUMMARY")
print("=" * 65)
print(metrics[["country","total_actual","total_forecast","total_variance_pct","MAE","MAPE"]].to_string(index=False))
print(f"\nAccuracy Breakdown:\n{df['accuracy_bucket'].value_counts().to_string()}")
print(f"\nModel Bias:\n{df['model_direction'].value_counts().to_string()}")

# Save summary CSV
metrics.to_csv(os.path.join(OUT_DIR, "reconciliation_summary.csv"), index=False)

# ─────────────────────────────────────────────
# CHART 1 — Forecast vs Actual per Country
# ─────────────────────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(16, 18), facecolor=BG_COLOR)
fig.suptitle("COVID-19  |  Forecast vs Actual Cases by Country (2021–2022)",
             fontsize=14, color=TEXT_COL, fontweight="bold", y=0.98)

for i, (country, ax) in enumerate(zip(df["country"].unique(), axes)):
    sub = df[df["country"] == country].sort_values("date")
    ax.fill_between(sub["date"], sub["actual_cases"],   alpha=0.25, color=PALETTE[i])
    ax.fill_between(sub["date"], sub["forecast_cases"], alpha=0.10, color="#FF6B6B")
    ax.plot(sub["date"], sub["actual_cases"],   color=PALETTE[i],  lw=1.8, label="Actual")
    ax.plot(sub["date"], sub["forecast_cases"], color="#FF6B6B",   lw=1.5, linestyle="--", label="Forecast")
    ax.set_title(f"  {country}", loc="left", fontsize=10, color=TEXT_COL, pad=4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K"))
    ax.grid(True, axis="y"); ax.legend(fontsize=8, framealpha=0, labelcolor=TEXT_COL, loc="upper right")
    ax.set_xlim(sub["date"].min(), sub["date"].max())

plt.tight_layout(rect=[0, 0, 1, 0.97])
p = os.path.join(OUT_DIR, "01_forecast_vs_actual.png")
plt.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG_COLOR); plt.close()
print(f"\nChart 1 saved: {p}")

# ─────────────────────────────────────────────
# CHART 2 — Variance % Heatmap
# ─────────────────────────────────────────────
pivot = df.groupby(["country", "year_quarter"])["variance_pct"].mean().unstack()
pivot = pivot[sorted(pivot.columns)]

fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG_COLOR)
fig.suptitle("Variance % Heatmap  |  Country x Quarter  (+ = Over-forecast, - = Under-forecast)",
             fontsize=12, color=TEXT_COL, fontweight="bold")
cmap = sns.diverging_palette(10, 133, as_cmap=True)
sns.heatmap(pivot, ax=ax, cmap=cmap, center=0, annot=True, fmt=".1f",
            linewidths=0.5, linecolor=GRID_COL, cbar_kws={"shrink": 0.6},
            annot_kws={"size": 8, "color": "white"})
ax.set_xlabel("Quarter", fontsize=9); ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
p = os.path.join(OUT_DIR, "02_variance_heatmap.png")
plt.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG_COLOR); plt.close()
print(f"Chart 2 saved: {p}")

# ─────────────────────────────────────────────
# CHART 3 — MAPE Bar + Accuracy Donut
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(16, 6), facecolor=BG_COLOR)
gs  = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1])

ax1 = fig.add_subplot(gs[0])
colors = [PALETTE[i] for i in range(len(metrics))]
bars = ax1.barh(metrics["country"], metrics["MAPE"], color=colors, height=0.5, edgecolor=GRID_COL)
ax1.set_title("MAPE by Country  (Mean Absolute Percentage Error)", loc="left", color=TEXT_COL)
ax1.set_xlabel("MAPE %")
ax1.axvline(x=10, color="#FFD166", lw=1.2, linestyle="--", alpha=0.7)
ax1.text(10.5, -0.5, "10% threshold", color="#FFD166", fontsize=8)
for bar, val in zip(bars, metrics["MAPE"]):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}%", va="center", fontsize=9, color=TEXT_COL)
ax1.grid(True, axis="x", alpha=0.3); ax1.set_facecolor(BG_COLOR)

ax2  = fig.add_subplot(gs[1])
acc  = df["accuracy_bucket"].value_counts()
donut_colors = ["#06D6A0", "#FFD166", "#FF9F43", "#FF6B6B"]
wedges, texts, autotexts = ax2.pie(
    acc.values, labels=None, autopct="%1.0f%%",
    colors=donut_colors[:len(acc)], startangle=90, pctdistance=0.75,
    wedgeprops={"width": 0.55, "edgecolor": BG_COLOR, "linewidth": 2}
)
for at in autotexts: at.set_fontsize(8); at.set_color(BG_COLOR)
ax2.legend(acc.index.tolist(), loc="lower center", bbox_to_anchor=(0.5, -0.12),
           fontsize=7.5, framealpha=0, labelcolor=TEXT_COL, ncol=2)
ax2.set_title("Accuracy Distribution", color=TEXT_COL)
plt.tight_layout()
p = os.path.join(OUT_DIR, "03_mape_accuracy.png")
plt.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG_COLOR); plt.close()
print(f"Chart 3 saved: {p}")

# ─────────────────────────────────────────────
# CHART 4 — Rolling MAE (Model Drift)
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG_COLOR)
fig.suptitle("Rolling 4-Week MAE  |  Model Accuracy Drift Over Time",
             fontsize=12, color=TEXT_COL, fontweight="bold")
for i, country in enumerate(df["country"].unique()):
    sub     = df[df["country"] == country].sort_values("date")
    rolling = sub.set_index("date")["abs_error"].rolling("28D").mean()
    ax.plot(rolling.index, rolling.values, color=PALETTE[i], lw=1.8, label=country)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K"))
ax.set_ylabel("Rolling MAE (cases)")
ax.legend(fontsize=9, framealpha=0, labelcolor=TEXT_COL)
ax.grid(True, alpha=0.3)
plt.tight_layout()
p = os.path.join(OUT_DIR, "04_rolling_mae.png")
plt.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG_COLOR); plt.close()
print(f"Chart 4 saved: {p}")

# ─────────────────────────────────────────────
# CHART 5 — Scatter: Forecast vs Actual
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 4), facecolor=BG_COLOR)
fig.suptitle("Scatter: Forecast vs Actual  |  Perfect Model = Points on Diagonal",
             fontsize=12, color=TEXT_COL, fontweight="bold")
for i, (country, ax) in enumerate(zip(df["country"].unique(), axes)):
    sub = df[df["country"] == country]
    ax.scatter(sub["actual_cases"], sub["forecast_cases"],
               color=PALETTE[i], alpha=0.55, s=18, edgecolors="none")
    mn = min(sub["actual_cases"].min(), sub["forecast_cases"].min())
    mx = max(sub["actual_cases"].max(), sub["forecast_cases"].max())
    ax.plot([mn, mx], [mn, mx], "--", color="white", lw=1, alpha=0.5)
    corr = sub["actual_cases"].corr(sub["forecast_cases"])
    ax.set_title(f"{country}\nr={corr:.2f}", color=TEXT_COL, fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K"))
    ax.set_facecolor(BG_COLOR); ax.grid(True, alpha=0.2)
plt.tight_layout()
p = os.path.join(OUT_DIR, "05_scatter_forecast_actual.png")
plt.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG_COLOR); plt.close()
print(f"Chart 5 saved: {p}")

print(f"\nAll outputs saved to: {os.path.abspath(OUT_DIR)}")


  RECONCILIATION SUMMARY
country  total_actual  total_forecast  total_variance_pct    MAE  MAPE
 Brazil      15165757        16075383                6.00  15216 10.43
Germany       6710712         8090875               20.57  13497 20.92
  India      52866227        63509382               20.13 104540 20.57
     UK       9264545        10373193               11.97  13123 14.73
    USA      36608576        42542217               16.21  59778 16.98

Accuracy Breakdown:
accuracy_bucket
Poor (15–30%)            181
Acceptable (5–15%)       168
Highly Accurate (≤5%)     97
Unacceptable (>30%)       74

Model Bias:
model_direction
Over-forecast     441
Under-forecast     79

Chart 1 saved: covid_reconciliation_data/outputs/01_forecast_vs_actual.png
Chart 2 saved: covid_reconciliation_data/outputs/02_variance_heatmap.png
Chart 3 saved: covid_reconciliation_data/outputs/03_mape_accuracy.png
Chart 4 saved: covid_reconciliation_data/outputs/04_rolling_mae.png
Chart 5 saved: covid_reconciliation

In [7]:
"""
COVID-19 Forecast vs Actuals Dataset Generator
Project: Recalculated Modelling Results vs Actual Outcomes

HOW TO RUN:
  - Google Colab / Jupyter: run as-is, files save to current working directory
  - Local machine:          same — or change OUTPUT_DIR below for a specific folder
"""

import pandas as pd
import numpy as np
import os

# ─────────────────────────────────────────────
# CONFIG — change OUTPUT_DIR if needed
# ─────────────────────────────────────────────
OUTPUT_DIR = "covid_reconciliation_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(42)

START_DATE = "2021-01-01"
END_DATE   = "2022-12-31"
dates      = pd.date_range(START_DATE, END_DATE, freq="W")

country_params = {
    "India":   (350000, 450000, 15000),
    "USA":     (250000, 300000, 12000),
    "UK":      ( 60000,  80000,  4000),
    "Brazil":  (100000, 130000,  6000),
    "Germany": ( 40000,  60000,  3000),
}

def simulate_actuals(n, peak1, peak2, noise_std):
    t     = np.linspace(0, 4 * np.pi, n)
    wave1 = peak1 * np.abs(np.sin(t * 0.6))
    wave2 = peak2 * np.abs(np.sin(t * 1.1 + 1.5))
    noise = np.random.normal(0, noise_std, n)
    return np.clip(wave1 + wave2 + noise, 0, None).astype(int)

rows = []
for country, (p1, p2, noise) in country_params.items():
    actuals   = simulate_actuals(len(dates), p1, p2, noise)
    bias      = np.random.uniform(0.80, 1.20)
    rand_err  = np.random.normal(1.0, 0.12, len(dates))
    forecasts = np.clip((actuals * bias * rand_err).astype(int), 0, None)
    for i, d in enumerate(dates):
        rows.append({
            "date":           d.strftime("%Y-%m-%d"),
            "country":        country,
            "forecast_cases": int(forecasts[i]),
            "actual_cases":   int(actuals[i]),
            "variance":       int(forecasts[i]) - int(actuals[i]),
            "variance_pct":   round(((int(forecasts[i]) - int(actuals[i])) / max(int(actuals[i]), 1)) * 100, 2),
            "abs_error":      abs(int(forecasts[i]) - int(actuals[i])),
        })

df = pd.DataFrame(rows)
df["date"]         = pd.to_datetime(df["date"])
df["year"]         = df["date"].dt.year
df["quarter"]      = df["date"].dt.quarter
df["month"]        = df["date"].dt.month
df["month_name"]   = df["date"].dt.strftime("%b")
df["year_quarter"] = df["year"].astype(str) + "-Q" + df["quarter"].astype(str)

def accuracy_bucket(pct):
    pct = abs(pct)
    if pct <= 5:    return "Highly Accurate (<=5%)"
    elif pct <= 15: return "Acceptable (5-15%)"
    elif pct <= 30: return "Poor (15-30%)"
    else:           return "Unacceptable (>30%)"

df["accuracy_bucket"] = df["variance_pct"].apply(accuracy_bucket)
df["model_direction"] = df["variance"].apply(
    lambda x: "Over-forecast" if x > 0 else ("Under-forecast" if x < 0 else "Exact")
)

out_path = os.path.join(OUTPUT_DIR, "covid_forecast_actuals.csv")
df.to_csv(out_path, index=False)
print(f"Saved to: {os.path.abspath(out_path)}")
print(f"Rows: {len(df):,}  |  Countries: {df['country'].nunique()}")
print(df.head(3).to_string())

Saved to: /content/covid_reconciliation_data/covid_forecast_actuals.csv
Rows: 520  |  Countries: 5
        date country  forecast_cases  actual_cases  variance  variance_pct  abs_error  year  quarter  month month_name year_quarter         accuracy_bucket model_direction
0 2021-01-03   India          551021        456323     94698         20.75      94698  2021        1      1        Jan      2021-Q1           Poor (15-30%)   Over-forecast
1 2021-01-10   India          421336        472619    -51283        -10.85      51283  2021        1      1        Jan      2021-Q1      Acceptable (5-15%)  Under-forecast
2 2021-01-17   India          524078        502016     22062          4.39      22062  2021        1      1        Jan      2021-Q1  Highly Accurate (<=5%)   Over-forecast


In [9]:
# ============================================================
# COVID-19 FORECAST RECONCILIATION — GOOGLE COLAB NOTEBOOK
# Project: Recalculated Modelling Results vs Actual Outcomes
#
# INSTRUCTIONS:
#   Copy each cell block into a separate Colab cell.
#   Run them top to bottom in order.
#   All charts display inline inside Colab automatically.
# ============================================================


# ─────────────────────────────────────────────────────────────
# CELL 1 — Install & Import Libraries
# ─────────────────────────────────────────────────────────────

!pip install seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries ready")


# ─────────────────────────────────────────────────────────────
# CELL 2 — Generate Dataset  (mirrors generate_data.py exactly)
# ─────────────────────────────────────────────────────────────

np.random.seed(42)

OUTPUT_DIR = "covid_reconciliation_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dates = pd.date_range("2021-01-01", "2022-12-31", freq="W")

country_params = {
    "India":   (350000, 450000, 15000),
    "USA":     (250000, 300000, 12000),
    "UK":      ( 60000,  80000,  4000),
    "Brazil":  (100000, 130000,  6000),
    "Germany": ( 40000,  60000,  3000),
}

def simulate_actuals(n, peak1, peak2, noise_std):
    t     = np.linspace(0, 4 * np.pi, n)
    wave1 = peak1 * np.abs(np.sin(t * 0.6))
    wave2 = peak2 * np.abs(np.sin(t * 1.1 + 1.5))
    noise = np.random.normal(0, noise_std, n)
    return np.clip(wave1 + wave2 + noise, 0, None).astype(int)

rows = []
for country, (p1, p2, noise) in country_params.items():
    actuals   = simulate_actuals(len(dates), p1, p2, noise)
    bias      = np.random.uniform(0.80, 1.20)
    rand_err  = np.random.normal(1.0, 0.12, len(dates))
    forecasts = np.clip((actuals * bias * rand_err).astype(int), 0, None)
    for i, d in enumerate(dates):
        rows.append({
            "date":           d.strftime("%Y-%m-%d"),
            "country":        country,
            "forecast_cases": int(forecasts[i]),
            "actual_cases":   int(actuals[i]),
            "variance":       int(forecasts[i]) - int(actuals[i]),
            "variance_pct":   round(((int(forecasts[i]) - int(actuals[i])) / max(int(actuals[i]), 1)) * 100, 2),
            "abs_error":      abs(int(forecasts[i]) - int(actuals[i])),
        })

df = pd.DataFrame(rows)
df["date"]         = pd.to_datetime(df["date"])
df["year"]         = df["date"].dt.year
df["quarter"]      = df["date"].dt.quarter
df["month"]        = df["date"].dt.month
df["month_name"]   = df["date"].dt.strftime("%b")
df["year_quarter"] = df["year"].astype(str) + "-Q" + df["quarter"].astype(str)

def accuracy_bucket(pct):
    pct = abs(pct)
    if pct <= 5:    return "Highly Accurate (<=5%)"
    elif pct <= 15: return "Acceptable (5-15%)"
    elif pct <= 30: return "Poor (15-30%)"
    else:           return "Unacceptable (>30%)"

df["accuracy_bucket"] = df["variance_pct"].apply(accuracy_bucket)
df["model_direction"] = df["variance"].apply(
    lambda x: "Over-forecast" if x > 0 else ("Under-forecast" if x < 0 else "Exact")
)

# Save CSV
csv_path = os.path.join(OUTPUT_DIR, "covid_forecast_actuals.csv")
df.to_csv(csv_path, index=False)

print(f"✅ Dataset ready — {len(df):,} rows across {df['country'].nunique()} countries")
print(f"   Saved to: {csv_path}")
print(f"\n   Columns: {list(df.columns)}")
print(f"\n   Sample:\n{df.head(3).to_string()}")


# ─────────────────────────────────────────────────────────────
# CELL 3 — Reconciliation Metrics Summary
# ─────────────────────────────────────────────────────────────

def reconciliation_metrics(df):
    grp = df.groupby("country").agg(
        total_actual      = ("actual_cases",   "sum"),
        total_forecast    = ("forecast_cases", "sum"),
        mean_abs_error    = ("abs_error",      "mean"),
        weeks             = ("date",           "count"),
    ).reset_index()
    grp["total_variance"]     = grp["total_forecast"] - grp["total_actual"]
    grp["total_variance_pct"] = ((grp["total_variance"] / grp["total_actual"]) * 100).round(2)
    grp["MAE"]                = grp["mean_abs_error"].round(0).astype(int)
    mape_series = (
        df.groupby("country")["abs_error"].mean() /
        df.groupby("country")["actual_cases"].mean() * 100
    ).round(2)
    grp["MAPE"] = grp["country"].map(mape_series)
    return grp

metrics = reconciliation_metrics(df)

print("=" * 65)
print("  RECONCILIATION SUMMARY — COVID FORECAST vs ACTUALS")
print("=" * 65)
print(metrics[["country","total_actual","total_forecast","total_variance_pct","MAE","MAPE"]].to_string(index=False))
print(f"\n  Accuracy Breakdown:\n{df['accuracy_bucket'].value_counts().to_string()}")
print(f"\n  Model Bias:\n{df['model_direction'].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────
# CELL 4 — Chart Style Setup  (run once, applies to all charts)
# ─────────────────────────────────────────────────────────────

PALETTE  = ["#00C2CB", "#FF6B6B", "#FFD166", "#06D6A0", "#A78BFA"]
BG_COLOR = "#0D1117"
TEXT_COL = "#E6EDF3"
GRID_COL = "#21262D"

plt.rcParams.update({
    "figure.facecolor": BG_COLOR,
    "axes.facecolor":   BG_COLOR,
    "axes.edgecolor":   GRID_COL,
    "axes.labelcolor":  TEXT_COL,
    "xtick.color":      TEXT_COL,
    "ytick.color":      TEXT_COL,
    "text.color":       TEXT_COL,
    "grid.color":       GRID_COL,
    "grid.linewidth":   0.5,
    "font.family":      "monospace",
    "axes.titlesize":   11,
    "axes.labelsize":   9,
})

print("✅ Chart style applied")


# ─────────────────────────────────────────────────────────────
# CELL 5 — CHART 1: Forecast vs Actual (All 5 Countries)
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(5, 1, figsize=(16, 20), facecolor=BG_COLOR)
fig.suptitle(
    "COVID-19  |  Weekly Forecast vs Actual Cases by Country (2021–2022)",
    fontsize=14, color=TEXT_COL, fontweight="bold", y=0.99
)

for i, (country, ax) in enumerate(zip(df["country"].unique(), axes)):
    sub = df[df["country"] == country].sort_values("date")

    # Shaded areas
    ax.fill_between(sub["date"], sub["actual_cases"],   alpha=0.20, color=PALETTE[i])
    ax.fill_between(sub["date"], sub["forecast_cases"], alpha=0.08, color="#FF6B6B")

    # Lines
    ax.plot(sub["date"], sub["actual_cases"],
            color=PALETTE[i], lw=2.0, label="Actual")
    ax.plot(sub["date"], sub["forecast_cases"],
            color="#FF6B6B", lw=1.5, linestyle="--", label="Forecast", alpha=0.85)

    # Formatting
    ax.set_title(f"  {country}", loc="left", fontsize=11,
                 color=PALETTE[i], pad=6, fontweight="bold")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K")
    )
    ax.grid(True, axis="y", alpha=0.4)
    ax.legend(fontsize=9, framealpha=0, labelcolor=TEXT_COL,
              loc="upper right", ncol=2)
    ax.set_xlim(sub["date"].min(), sub["date"].max())

    # MAPE annotation
    mape_val = metrics.loc[metrics["country"] == country, "MAPE"].values[0]
    ax.annotate(
        f"MAPE: {mape_val:.1f}%",
        xy=(0.01, 0.85), xycoords="axes fraction",
        fontsize=9, color=TEXT_COL,
        bbox=dict(boxstyle="round,pad=0.3", fc=GRID_COL, alpha=0.7, ec="none")
    )

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig(os.path.join(OUTPUT_DIR, "01_forecast_vs_actual.png"),
            dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()
print("✅ Chart 1 displayed and saved")


# ─────────────────────────────────────────────────────────────
# CELL 6 — CHART 2: Variance % Heatmap (Country × Quarter)
# ─────────────────────────────────────────────────────────────

pivot = df.groupby(["country", "year_quarter"])["variance_pct"].mean().unstack()
pivot = pivot[sorted(pivot.columns)]

fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG_COLOR)
fig.suptitle(
    "Variance % Heatmap  |  Country × Quarter\n"
    "Red = Model Over-forecast  |  Green = Model Under-forecast",
    fontsize=12, color=TEXT_COL, fontweight="bold"
)

cmap = sns.diverging_palette(10, 133, as_cmap=True)

sns.heatmap(
    pivot, ax=ax,
    cmap=cmap,
    center=0,
    annot=True,
    fmt=".1f",
    linewidths=0.6,
    linecolor=GRID_COL,
    cbar_kws={"shrink": 0.6, "label": "Avg Variance %"},
    annot_kws={"size": 9, "color": "white", "fontweight": "bold"}
)

ax.set_xlabel("Quarter", fontsize=10, labelpad=8)
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_variance_heatmap.png"),
            dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()
print("✅ Chart 2 displayed and saved")


# ─────────────────────────────────────────────────────────────
# CELL 7 — CHART 3: Scatter — Forecast vs Actual per Country
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 5, figsize=(20, 5), facecolor=BG_COLOR)
fig.suptitle(
    "Scatter: Forecast vs Actual Cases  |  Dashed line = Perfect Model  |  r = Pearson Correlation",
    fontsize=12, color=TEXT_COL, fontweight="bold"
)

for i, (country, ax) in enumerate(zip(df["country"].unique(), axes)):
    sub = df[df["country"] == country]

    # Scatter points
    ax.scatter(
        sub["actual_cases"], sub["forecast_cases"],
        color=PALETTE[i], alpha=0.55, s=22, edgecolors="none"
    )

    # Perfect forecast line (diagonal)
    mn = min(sub["actual_cases"].min(), sub["forecast_cases"].min())
    mx = max(sub["actual_cases"].max(), sub["forecast_cases"].max())
    ax.plot([mn, mx], [mn, mx], "--", color="white", lw=1.2, alpha=0.6, label="Perfect model")

    # Correlation
    corr = sub["actual_cases"].corr(sub["forecast_cases"])
    mape = metrics.loc[metrics["country"] == country, "MAPE"].values[0]

    ax.set_title(
        f"{country}\nr = {corr:.2f}  |  MAPE = {mape:.1f}%",
        color=TEXT_COL, fontsize=9, pad=6
    )
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K")
    )
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K")
    )
    ax.set_xlabel("Actual cases", fontsize=8)
    ax.set_ylabel("Forecast cases", fontsize=8)
    ax.set_facecolor(BG_COLOR)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_scatter_forecast_actual.png"),
            dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()
print("✅ Chart 3 displayed and saved")


# ─────────────────────────────────────────────────────────────
# CELL 8 — CHART 4: Rolling 4-Week MAE (Model Drift Over Time)
# ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(16, 6), facecolor=BG_COLOR)
fig.suptitle(
    "Rolling 4-Week MAE  |  Tracking Model Accuracy Drift Over Time\n"
    "Rising line = model getting worse  |  Flat line = stable accuracy",
    fontsize=12, color=TEXT_COL, fontweight="bold"
)

for i, country in enumerate(df["country"].unique()):
    sub     = df[df["country"] == country].sort_values("date")
    rolling = sub.set_index("date")["abs_error"].rolling("28D").mean()

    ax.plot(
        rolling.index, rolling.values,
        color=PALETTE[i], lw=2.0, label=country
    )
    # Annotate end point
    last_x = rolling.index[-1]
    last_y = rolling.values[-1]
    ax.annotate(
        f"  {country}",
        xy=(last_x, last_y),
        fontsize=8, color=PALETTE[i], va="center"
    )

ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}K")
)
ax.set_ylabel("Rolling 4-Week MAE (cases)", fontsize=10)
ax.set_xlabel("Date", fontsize=10)
ax.legend(fontsize=9, framealpha=0, labelcolor=TEXT_COL, loc="upper right")
ax.grid(True, alpha=0.3)
ax.set_xlim(df["date"].min(), df["date"].max())

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_rolling_mae.png"),
            dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()
print("✅ Chart 4 displayed and saved")


# ─────────────────────────────────────────────────────────────
# CELL 9 — Download all saved charts from Colab (optional)
# ─────────────────────────────────────────────────────────────

from google.colab import files

chart_files = [
    os.path.join(OUTPUT_DIR, "01_forecast_vs_actual.png"),
    os.path.join(OUTPUT_DIR, "02_variance_heatmap.png"),
    os.path.join(OUTPUT_DIR, "05_scatter_forecast_actual.png"),
    os.path.join(OUTPUT_DIR, "04_rolling_mae.png"),
    os.path.join(OUTPUT_DIR, "covid_forecast_actuals.csv"),
]

for f in chart_files:
    if os.path.exists(f):
        files.download(f)
        print(f"⬇️  Downloading: {f}")
    else:
        print(f"⚠️  Not found: {f}")

✅ Libraries ready
✅ Dataset ready — 520 rows across 5 countries
   Saved to: covid_reconciliation_data/covid_forecast_actuals.csv

   Columns: ['date', 'country', 'forecast_cases', 'actual_cases', 'variance', 'variance_pct', 'abs_error', 'year', 'quarter', 'month', 'month_name', 'year_quarter', 'accuracy_bucket', 'model_direction']

   Sample:
        date country  forecast_cases  actual_cases  variance  variance_pct  abs_error  year  quarter  month month_name year_quarter         accuracy_bucket model_direction
0 2021-01-03   India          551021        456323     94698         20.75      94698  2021        1      1        Jan      2021-Q1           Poor (15-30%)   Over-forecast
1 2021-01-10   India          421336        472619    -51283        -10.85      51283  2021        1      1        Jan      2021-Q1      Acceptable (5-15%)  Under-forecast
2 2021-01-17   India          524078        502016     22062          4.39      22062  2021        1      1        Jan      2021-Q1  Highl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading: covid_reconciliation_data/01_forecast_vs_actual.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

⬇️  Downloading: covid_reconciliation_data/04_rolling_mae.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading: covid_reconciliation_data/covid_forecast_actuals.csv
